# 第十轮：神经群动力学与局部突触敏态
本轮按 BIOLOGICAL_CONSTRAINTS.md 测试输入结束后是否保留频带、逐段时长和顺序。模型内部只有神经元活动、适应、突触前痕迹和短期突触状态；没有事件表、位置键或查询。训练期诊断头直接读取最终神经群活动，暂不训练自主发声。


In [ ]:
from pathlib import Path
import subprocess
repo = Path('/content/Parrot')
if not repo.exists():
    subprocess.run(['git', 'clone', 'https://github.com/DoctorAwe/Parrot.git', str(repo)], check=True)
else:
    subprocess.run(['git', '-C', str(repo), 'pull', '--ff-only'], check=True)
commit = subprocess.check_output(['git', '-C', str(repo), 'rev-parse', '--short', 'HEAD'], text=True).strip()
print('代码版本:', commit)
%cd /content/Parrot


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
root = Path('/content/drive/MyDrive/Parrot')
out = root / 'trial10_results'
out.mkdir(parents=True, exist_ok=True)


## 快速检查


In [ ]:
!python trial10.py --smoke --batch 4 --out /content/Parrot/trial10_smoke


## 正式训练与评估
分别从头训练有局部短期突触敏态和固定突触对照。评估 2、4、6 段、等待 0/16/64 帧、重复频带和逐段时长交换。若耗时较长，请保留 Drive 中日志与错误信息。


In [ ]:
!python trial10.py --steps 2200 --batch 32 --out {out}


In [ ]:
import json
path = out / 'metrics.json'
report = json.loads(path.read_text(encoding='utf-8'))
report['git_commit'] = commit
path.write_text(json.dumps(report, indent=2), encoding='utf-8')
for model, groups in report.items():
    if model == 'git_commit': continue
    print(model)
    for key in ('n2_wait0','n4_wait16','n6_wait64','duration_swap_pairs'):
        print(key, groups[key])


## 回传
将 ZIP 放入项目 `result/` 目录。


In [ ]:
import shutil
archive = shutil.make_archive(str(root / 'trial10_results'), 'zip', root_dir=str(root), base_dir='trial10_results')
print(archive)
